# Atom-level knowledge graph

This notebook builds and explores the atom-level knowledge graph for the WGBO
corpus. Each Dutch Civil Code article is decomposed into atoms (the smallest
units carrying a single legal meaning), annotated with a nine-label schema,
and connected into a graph through three edge channels: shared annotation
tags, IDF-weighted overlaps, and semantic similarity.



Section flow:

1. Load the atom table
2. Corpus overview
3. Cross-article edges
4. Article-level connectivity
5. Tag weighting (IDF)
6. Semantic similarity
7. Legal model comparison
8. Atom classifier and evaluation
9. Legal hierarchy (placing atoms in the Dutch civil code tree?)


## 1. Load the atom table


In [1]:
import sys
import os
from pathlib import Path


def find_project_root(marker='src/data/atom_table.py'):
    for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
        if (p / marker).exists():
            return p
    raise RuntimeError(f"Could not find project root (no {marker} in any parent)")


PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)

import pandas as pd
pd.set_option('display.max_colwidth', 70)
pd.set_option('display.width', 220)

from src.data.atom_table import (
    build_atom_table, summary_stats, tag_frequencies, show_atom,
    build_edge_table, add_edge_summary, article_connectivity,
    compute_tag_weights, show_tag_weights,
    compute_tag_similarities, build_semantic_edge_table, unique_tags_by_field,
    classify_atom, evaluate_classifier_loo,
    LIST_FIELDS,
)

df = build_atom_table()
print(f'Loaded {len(df)} atoms across {df["article_id"].nunique()} articles')
df.head()


Loaded 61 atoms across 22 articles


,atom_id,article_id,lid,condition_type,text,text_nl,actors,legal_relations,acts,geographical_domain,temporal,explicit_references,hierarchies,residual,source,annotator,annotated_at,notes
0,6:162(1).a,6:162,1,post_condition,"One who commits an unlawful act against another, which can be attr...","Hij die jegens een ander een onrechtmatige daad pleegt, welke hem ...","[wrongdoer, injured party]",[],"[commits, compensates]",[],[],[],[],"[unlawful act, damage, attribution]","Dutch Civil Code, Book 6, Title 6.3.1",Claude-draft,2026-07-01,Central tort rule — parallel to 6:74's central contract rule. Dama...
1,6:162(2).a,6:162,2,post_condition,An unlawful act is deemed to be an infringement of a right and an ...,Als onrechtmatige daad worden aangemerkt een inbreuk op een recht ...,[],[],[],[],[],[],[],"[unlawful act, infringement, right, act, omission, legal duty, unw...","Dutch Civil Code, Book 6, Title 6.3.1",Claude-draft,2026-07-01,Definitional atom for 'unlawful act'. AND-connected clauses within...
2,6:162(2).b,6:162,2,pre_condition,all this subject to the presence of a ground for justification,een en ander behoudens de aanwezigheid van een rechtvaardigingsgrond,[],[],[],[],[],[],[subject to the presence of a ground for justification],"[justification, ground]","Dutch Civil Code, Book 6, Title 6.3.1",Claude-draft,2026-07-01,Hierarchy tag captures 'subject to' — parallels 7:454(3).a treatme...
3,6:162(3).a,6:162,3,post_condition,An unlawful act can be attributed to the perpetrator if it is due ...,"Een onrechtmatige daad kan aan de dader worden toegerekend, indien...",[wrongdoer],[],[],[],[],[],[],"[unlawful act, attribution, fault, law, generally accepted views, ...","Dutch Civil Code, Book 6, Title 6.3.1",Claude-draft,2026-07-01,Attribution rule for tort. Note shared residuals with 6:75 ('fault...
4,6:170(1).a,6:170,1,post_condition,"For damage caused to a third party by a fault of a subordinate, th...","Voor schade, aan een derde toegebracht door een fout van een onder...","[employer, subordinate, third party]",[service relationship],[],[],[],[],[],"[damage, fault, liability, task]","Dutch Civil Code, Book 6, Title 6.3.2",Claude-draft,2026-07-01,Tort-based vicarious liability. Employer/subordinate/third party i...


## 2. Corpus overview

Per-article summary followed by tag frequencies for each schema field.

Lists all articles and their attributes (number of : atoms, post-conditions, pre-conditions, legal relations, acts, geographical domain, temporal, explicit references, hierarchies, residual).


In [2]:
summary_stats(df)


,article_id,n_atoms,n_pre,n_post,actors,legal_relations,acts,geographical_domain,temporal,explicit_references,hierarchies,residual
0,6:162,4,1,3,2,0,1,0,0,0,1,4
1,6:170,3,2,1,3,2,0,0,0,0,0,3
2,6:74,3,1,2,3,1,1,0,0,1,0,3
3,6:75,1,0,1,1,0,0,0,0,0,0,1
4,6:76,1,0,1,1,1,1,0,0,0,0,1
5,7:454,8,2,6,7,2,6,0,3,1,1,8
6,7:455,3,2,1,2,1,1,0,1,1,2,3
7,synth:1,2,1,1,2,0,2,2,1,0,0,1
8,synth:10,3,2,1,2,0,3,0,0,0,1,3
9,synth:11,3,2,1,3,1,2,0,1,0,0,3


In [3]:
for field in LIST_FIELDS:
    freqs = tag_frequencies(df, field)
    if len(freqs) == 0:
        continue
    print(f'\n--- {field} ---')
    print(freqs.to_string())



--- actors ---
actors
patient                                   17
care provider                              7
debtor                                     5
colonist                                   5
subordinate                                3
employer                                   3
citizen                                    3
unemancipated minor                        2
wrongdoer                                  2
injured party                              1
third party                                1
someone other than patient                 1
creditor                                   1
auxiliary person                           1
certified emergency medical technician     1
attending physician                        1
lead epidemiologist                        1
orbital health authority                   1
legal guardian                             1
colonial medical board                     1
automated triage system                    1
treating specialist             

## 3. Cross-article edges

One row per pair of atoms from different articles that share at least one tag
value in the same field. Each edge is one overlapping tag. Baseline strength
is the count of shared tags in that field.

Overview of 'Exact Match' edges, edges that are formed purely through a shared value within a field (field being actors, legal relations, residuals etc...)


In [4]:
edges = build_edge_table(df)
print(f'{len(edges)} cross-article edges')
print()
print('Edges by field:')
print(edges['field'].value_counts().to_string())
edges


176 cross-article edges

Edges by field:
field
actors                 139
residual                29
legal_relations          3
acts                     2
temporal                 2
explicit_references      1


,atom_a,atom_b,article_a,article_b,field,shared,strength,weighted_strength
0,6:162(1).a,6:170(1).a,6:162,6:170,residual,[damage],1,1.0
1,6:162(1).a,6:74(1).a,6:162,6:74,acts,[compensates],1,1.0
2,6:162(1).a,6:74(1).a,6:162,6:74,residual,[damage],1,1.0
3,6:162(1).a,6:74(1).b,6:162,6:74,residual,[attribution],1,1.0
4,6:162(1).a,6:75(1).a,6:162,6:75,residual,[attribution],1,1.0
...,...,...,...,...,...,...,...,...
171,synth:11(1).a,synth:13(1).c,synth:11,synth:13,actors,[patient],1,1.0
172,synth:11(1).b,synth:13(1).c,synth:11,synth:13,actors,[patient],1,1.0
173,synth:11(1).c,synth:13(1).c,synth:11,synth:13,actors,[patient],1,1.0
174,synth:13(1).a,synth:15(1).a,synth:13,synth:15,actors,[colonist],1,1.0


### Atoms as hubs and isolates

For each atom, the number of edges it has and which schema fields produce
them. High `n_edges` = hub. Zero edges = the atom shares no tags with any
other article's atoms.


In [5]:
df_e = add_edge_summary(df, edges)
df_e.sort_values('n_edges', ascending=False)[
    ['atom_id', 'n_edges', 'connected_atoms', 'edge_fields']
]


,atom_id,n_edges,connected_atoms,edge_fields
20,7:455(1).a,24,"[7:454(1).a, 7:454(1).b, 7:454(1).c, 7:454(1).d, 7:454(1).e, 7:454...","[actors, legal_relations, residual]"
21,7:455(2).a,19,"[6:74(2).a, 7:454(1).a, 7:454(1).b, 7:454(1).c, 7:454(1).d, 7:454(...","[actors, explicit_references, residual]"
55,synth:13(1).c,17,"[7:454(1).a, 7:454(1).b, 7:454(1).c, 7:454(1).e, 7:454(2).a, 7:455...","[actors, temporal]"
23,synth:1(1).a,16,"[7:454(1).a, 7:454(1).b, 7:454(1).c, 7:454(1).e, 7:454(2).a, 7:455...","[actors, temporal]"
17,7:454(2).a,15,"[7:455(1).a, 7:455(2).a, synth:1(1).a, synth:1(1).b, synth:11(1).a...","[actors, legal_relations, residual]"
...,...,...,...,...
47,synth:10(1).c,0,[],[]
56,synth:14(1).a,0,[],[]
54,synth:13(1).b,0,[],[]
57,synth:14(1).b,0,[],[]


In [6]:
df_e.loc[df_e['n_edges'] == 0, ['atom_id', 'condition_type', 'text']]


,atom_id,condition_type,text
1,6:162(2).a,post_condition,An unlawful act is deemed to be an infringement of a right and an ...
2,6:162(2).b,pre_condition,all this subject to the presence of a ground for justification
19,7:454(3).b,pre_condition,or as much longer as reasonably follows from the duty of a good ca...
25,synth:2(1).a,pre_condition,If a medical emergency occurs during an extravehicular surface tra...
26,synth:2(1).b,post_condition,any certified emergency medical technician may administer autonomo...
28,synth:3(1).b,post_condition,the orbital health authority must purge the specified data from al...
29,synth:4(1).a,pre_condition,If a pathogenic mutation is detected within a hydroponic agricultu...
31,synth:5(1).a,pre_condition,If an unemancipated minor undergoes artificial gravity acclimation...
32,synth:5(1).b,post_condition,the legal guardian may observe the physiological telemetry via a s...
35,synth:6(1).c,pre_condition,unless the pathogen is legally classified as a colony-threatening ...


## 4. Article-level connectivity

Aggregate atom-level edges up to article pairs. Useful for a first look at
which articles cluster together at the coarse level.

Overview of total edges across different *articles* and their shared fields.


In [7]:
article_connectivity(edges)


,article_a,article_b,n_edges,fields
0,6:162,6:170,4,[residual]
1,6:162,6:74,4,"[acts, residual]"
2,6:162,6:75,2,[residual]
3,6:162,7:455,1,[residual]
4,6:170,6:74,1,[residual]
5,6:170,6:75,3,[residual]
6,6:170,6:76,2,[residual]
7,6:74,6:75,5,"[actors, residual]"
8,6:74,6:76,6,"[actors, legal_relations, residual]"
9,6:74,7:455,1,[explicit_references]


## 5. Tag weighting

Common tags (`patient`, `care provider`) appear in most atoms and carry
little discriminative signal. IDF down-weights them and up-weights rare
tags. Two modes: `idf` (default, log inverse document frequency) and `tf`
(raw frequency).

Lists bottom 8 tags ( words or phrases that are very common they appear frequently across corpus so does not help in inferring knowledge).

Second output lists edges with the highest weights ie edges formed across tags that are lesser common accross the corpus.


In [8]:
weights_idf = compute_tag_weights(df, method='idf')
print(f'Computed {len(weights_idf)} tag weights')
print()
print('Top 8 (most discriminative):')
print(show_tag_weights(weights_idf).head(8).to_string())
print()
print('Bottom 8 (least discriminative):')
print(show_tag_weights(weights_idf).tail(8).to_string())


Computed 241 tag weights

Top 8 (most discriminative):
    field                                     tag  weight
0  actors                           injured party   3.434
1  actors                             third party   3.434
2  actors                                creditor   3.434
3  actors                        auxiliary person   3.434
4  actors                     lead epidemiologist   3.434
5  actors                orbital health authority   3.434
6  actors  certified emergency medical technician   3.434
7  actors                     attending physician   3.434

Bottom 8 (least discriminative):
        field            tag  weight
233  residual    attribution  2.5177
234  residual           data  2.5177
235    actors         debtor  2.3354
236    actors       colonist  2.3354
237  residual          fault  2.3354
238  residual           file  2.1812
239    actors  care provider  2.0477
240    actors        patient  1.2368


In [30]:
edges_weighted = build_edge_table(df, weights=weights_idf)

print('Top 5 by uniform strength (count of shared tags):')
print(edges_weighted.sort_values('strength', ascending=False).head(5)[
    ['atom_a', 'atom_b', 'shared', 'strength']
].to_string(index=False))
print()
print('Top 5 by IDF-weighted strength:')
print(edges_weighted.sort_values('weighted_strength', ascending=False).head(5)[
    ['atom_a', 'atom_b', 'shared', 'weighted_strength']
].to_string(index=False))


Top 5 by uniform strength (count of shared tags):
    atom_a     atom_b                                                        shared  strength
6:162(3).a  6:75(1).a [attribution, commerce, fault, generally accepted views, law]         5
7:454(1).b 7:455(1).a                                                  [data, file]         2
7:454(1).a 7:455(1).a                                      [care provider, patient]         2
 6:74(1).b  6:75(1).a                             [attribution, failure to perform]         2
7:454(1).b 7:455(1).a                                      [care provider, patient]         2

Top 5 by IDF-weighted strength:
    atom_a     atom_b                                                        shared  weighted_strength
6:162(3).a  6:75(1).a [attribution, commerce, fault, generally accepted views, law]            13.6510
 6:74(1).b  6:75(1).a                             [attribution, failure to perform]             5.2585
7:454(2).a 7:455(1).a                       

## 6. Semantic similarity

Exact set intersection misses synonyms and related concepts. The semantic
layer computes cosine similarity between every pair of unique tag values in
each field, using a sentence-transformer. Pairs above the threshold become
implicit tag equivalences, which then promote to atom-level edges.

Expanding on exact matches by adding semantic similarity to form edges across field values that are also similar enough to form an edge, 'enough' determined by threshold currently set to 0.55.
Still Unsure what model works best LegalBert did not preform as well as I expected so switched back to mpnet for now.

Lists similarity pairs and number of semantic edges


In [10]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('sentence-transformers/all-mpnet-base-v2') #not sure which to use still
sim_pairs = compute_tag_similarities(df, threshold=0.55, model=model)
sem_edges = build_semantic_edge_table(df, sim_pairs)

print(f'{len(sim_pairs)} high-similarity tag pairs')
print(f'{len(sem_edges)} semantic edges across atoms')
sim_pairs.head(10)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

45 high-similarity tag pairs
67 semantic edges across atoms


,field,tag_a,tag_b,similarity
0,actors,creditor,debtor,0.8954
1,explicit_references,paragraph 1,second paragraph,0.8022
2,residual,active medical emergency,medical emergency,0.7837
3,residual,civilian life-support telemetry,physiological telemetry,0.7407
4,temporal,after request,upon request,0.7242
5,residual,mandatory,necessary,0.7215
6,temporal,for twenty years,longer than twenty years,0.7192
7,actors,patient,someone other than patient,0.7153
8,actors,care provider,designated primary caregiver,0.7089
9,residual,physiological symptoms,symptoms,0.7069


### Exact vs semantic coverage

Both channels operate on the same atoms. Exact edges are deterministic and
auditable; semantic edges catch synonyms exact matching misses.

Summary/Comparison of Semantic and Exact Edges.


In [11]:
# Combine both channels in one table
exact = edges.copy()
exact['edge_kind'] = 'exact'

sem = sem_edges.copy()
sem['edge_kind'] = 'semantic'
sem['shared'] = sem['matched_pairs']
sem['strength'] = sem['n_matches']
sem['weighted_strength'] = sem['max_similarity']

cols = ['edge_kind', 'atom_a', 'atom_b', 'field', 'shared', 'strength', 'weighted_strength']
combined = pd.concat([exact[cols], sem[cols]], ignore_index=True)

# Coverage overlap
exact_pairs = {(r.atom_a, r.atom_b) for _, r in exact.iterrows()}
sem_pairs_set = {(r.atom_a, r.atom_b) for _, r in sem.iterrows()}
sem_only = sem_pairs_set - exact_pairs
exact_only = exact_pairs - sem_pairs_set
both = sem_pairs_set & exact_pairs

print(f'Exact edges:                {len(exact)}')
print(f'Semantic edges:             {len(sem_edges)}')
print(f'Atom pairs, exact only:     {len(exact_only)}')
print(f'Atom pairs, semantic only:  {len(sem_only)}')
print(f'Atom pairs in both:         {len(both)}')

if sem_only:
    print()
    print('Atom pairs connected ONLY semantically (no exact tag overlap):')
    for a, b in sorted(sem_only):
        print(f'  {a} <-> {b}')


Exact edges:                176
Semantic edges:             67
Atom pairs, exact only:     129
Atom pairs, semantic only:  34
Atom pairs in both:         32

Atom pairs connected ONLY semantically (no exact tag overlap):
  6:162(1).a <-> 7:454(3).a
  6:162(1).a <-> 7:455(2).b
  6:162(2).a <-> 6:75(1).a
  6:162(2).a <-> 7:454(3).b
  6:162(2).a <-> 7:455(2).b
  6:170(1).a <-> 7:455(2).b
  6:74(1).a <-> 7:455(2).b
  7:454(1).b <-> synth:3(1).b
  7:454(1).d <-> synth:11(1).b
  7:454(1).d <-> synth:3(1).b
  7:454(1).e <-> synth:12(1).b
  7:454(3).a <-> 7:455(2).a
  7:454(3).a <-> synth:11(1).b
  7:454(3).a <-> synth:15(1).b
  7:455(1).a <-> synth:10(1).b
  7:455(1).a <-> synth:3(1).b
  7:455(2).a <-> synth:3(1).b
  synth:10(1).c <-> synth:15(1).a
  synth:13(1).c <-> synth:14(1).b
  synth:2(1).a <-> synth:14(1).a
  synth:2(1).b <-> synth:12(1).a
  synth:3(1).a <-> synth:10(1).c
  synth:3(1).b <-> synth:9(1).b
  synth:4(1).a <-> synth:6(1).c
  synth:4(1).b <-> synth:12(1).b
  synth:4(1).b <->

## 7. Legal model comparison

Section 6 uses `all-mpnet-base-v2` a general-purpose English model. Are
legal-domain models better? Three candidates at the same threshold:

- **all-mpnet-base-v2** : general English baseline (~420 MB) 45 pairs at threshold 0.55  45 pairs at threshold 0.55
- **legal-bert-base-uncased** (nlpaueb) : Chalkidis et al.'s Legal-BERT (~440 MB) 7741 pairs at threshold 0.55
- **legal-xlm-roberta-base** (joelniklaus) : multilingual legal, covers Dutch (~1.1 GB) 4954 pairs at threshold 0.55

First run downloads ~2 GB total. Cached afterward.


legal-bert may require some filtering which will make it perform better maybe?




In [12]:
LEGAL_MODELS = {
    'baseline_mpnet':    'sentence-transformers/all-mpnet-base-v2',
    'legal_bert_en':     'nlpaueb/legal-bert-base-uncased',
    'legal_xlm_roberta': 'joelniklaus/legal-xlm-roberta-base',
}
COMPARISON_THRESHOLD = 0.55

import gc
sim_pairs_by_model = {}

for label, model_id in LEGAL_MODELS.items():
    print(f'\n=== {label} ({model_id}) ===')
    m = None
    try:
        m = SentenceTransformer(model_id)
        pairs = compute_tag_similarities(df, threshold=COMPARISON_THRESHOLD, model=m)
        sim_pairs_by_model[label] = pairs
        print(f'  {len(pairs)} pairs at threshold {COMPARISON_THRESHOLD}')
    except Exception as e:
        print(f'  Failed: {type(e).__name__}: {e}')
        sim_pairs_by_model[label] = None
    finally:
        if m is not None:
            del m
        gc.collect()



=== baseline_mpnet (sentence-transformers/all-mpnet-base-v2) ===


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

  45 pairs at threshold 0.55

=== legal_bert_en (nlpaueb/legal-bert-base-uncased) ===


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: nlpaueb/legal-bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  7741 pairs at threshold 0.55

=== legal_xlm_roberta (joelniklaus/legal-xlm-roberta-base) ===


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: joelniklaus/legal-xlm-roberta-base
Key                       | Status     | 
--------------------------+------------+-
lm_head.dense.bias        | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
pooler.dense.bias         | MISSING    | 
pooler.dense.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


  4954 pairs at threshold 0.55


### 7.1 Summary  pair counts and similarity stats


In [13]:
summary = []
for label, pairs in sim_pairs_by_model.items():
    if pairs is None:
        summary.append({'model': label, 'status': 'failed', 'n_pairs': 0,
                        'avg_similarity': None, 'max_similarity': None})
    else:
        summary.append({
            'model': label,
            'status': 'ok',
            'n_pairs': len(pairs),
            'avg_similarity': round(pairs['similarity'].mean(), 3) if len(pairs) else None,
            'max_similarity': round(pairs['similarity'].max(), 3) if len(pairs) else None,
        })
pd.DataFrame(summary)


,model,status,n_pairs,avg_similarity,max_similarity
0,baseline_mpnet,ok,45,0.642,0.895
1,legal_bert_en,ok,7741,0.733,0.987
2,legal_xlm_roberta,ok,4954,0.723,0.994


### 7.2 Top pairs per model




In [14]:
for label, pairs in sim_pairs_by_model.items():
    if pairs is None or len(pairs) == 0:
        print(f'\n=== {label}: no pairs ===')
        continue
    print(f'\n=== {label} — top 10 pairs ===')
    print(pairs.head(10).to_string(index=False))



=== baseline_mpnet — top 10 pairs ===
              field                           tag_a                        tag_b  similarity
             actors                        creditor                       debtor      0.8954
explicit_references                     paragraph 1             second paragraph      0.8022
           residual        active medical emergency            medical emergency      0.7837
           residual civilian life-support telemetry      physiological telemetry      0.7407
           temporal                   after request                 upon request      0.7242
           residual                       mandatory                    necessary      0.7215
           temporal                for twenty years     longer than twenty years      0.7192
             actors                         patient   someone other than patient      0.7153
             actors                   care provider designated primary caregiver      0.7089
           residual          ph

### 7.3 Pairs unique to each model


In [15]:
def pair_key(row):
    a, b = sorted([row['tag_a'], row['tag_b']])
    return (row['field'], a, b)

model_pairs_sets = {}
for label, pairs in sim_pairs_by_model.items():
    if pairs is None:
        continue
    model_pairs_sets[label] = {pair_key(r) for _, r in pairs.iterrows()}

for label, this_set in model_pairs_sets.items():
    others = set().union(*(v for k, v in model_pairs_sets.items() if k != label))
    only = this_set - others
    print(f'\n=== Only in {label} ({len(only)}) ===')
    pdf = sim_pairs_by_model[label]
    for field, a, b in list(only)[:8]:
        mask = ((pdf['field'] == field) &
                (((pdf['tag_a'] == a) & (pdf['tag_b'] == b)) |
                 ((pdf['tag_a'] == b) & (pdf['tag_b'] == a))))
        sim = pdf[mask]['similarity'].iloc[0] if mask.any() else None
        print(f'  {field}: {a!r} ~ {b!r}  (sim={sim:.3f})')



=== Only in baseline_mpnet (1) ===
  residual: 'definitively categorized' ~ 'legally classified'  (sim=0.592)

=== Only in legal_bert_en (3121) ===
  residual: 'certified' ~ 'surplus biological material'  (sim=0.811)
  residual: 'compromised network nodes' ~ 'incapacitated'  (sim=0.766)
  residual: 'definitively categorized' ~ 'health'  (sim=0.754)
  residual: 'electronic' ~ 'guaranteed mechanical mobility'  (sim=0.790)
  residual: 'good care provider' ~ 'task'  (sim=0.655)
  actors: 'assigned quarantine officers' ~ 'lead neurosurgeon'  (sim=0.801)
  residual: 'certified' ~ 'medical data breach'  (sim=0.784)
  residual: 'request' ~ 'surplus biological material'  (sim=0.787)

=== Only in legal_xlm_roberta (337) ===
  actors: 'Extraterrestrial Health Directorate' ~ 'wrongdoer'  (sim=0.568)
  residual: 'gravitational adaptation biometrics' ~ 'substantial interest'  (sim=0.634)
  residual: 'emergency retrieval drone' ~ 'legal duty'  (sim=0.660)
  residual: 'medical emergency' ~ 'specified

### 7.4 Downstream impact, Leave-One_Out (LOO) Cross validation accuracy per model

Surface pair counts are only interesting if they translate to classification
accuracy. Run leave-one-out with each model's `sim_pairs` and compare top-1,
top-3, MRR.

LOO : Isolate one atom , train model on the rest. Have model classify isolated and test how accurately it does so.

top-1 :  measures a system's ability to return the correct or most relevant item at the very first position.

top-3 :  measures a system's ability to return the correct or most relevant item within top 3 positions.


In [16]:
downstream = []
for label, pairs in sim_pairs_by_model.items():
    if pairs is None:
        downstream.append({'model': label, 'status': 'skipped'})
        continue
    _, s = evaluate_classifier_loo(df, weights=weights_idf, sim_pairs=pairs, alpha=0.5, top_k=3)
    downstream.append({'model': label, 'n_sim_pairs': len(pairs), **s})

pd.DataFrame(downstream)


,model,n_sim_pairs,n_atoms,top1_accuracy,top3_accuracy,mean_reciprocal_rank
0,baseline_mpnet,45,61,0.2787,0.4262,0.3497
1,legal_bert_en,7741,61,0.1803,0.4426,0.3033
2,legal_xlm_roberta,4954,61,0.1967,0.4098,0.2951


### 7.5 Reading the results

Decision rule:

- More pairs AND higher top-1 accuracy → straight win, adopt as default
- More pairs but worse accuracy → the extra pairs are noise, stick with baseline
- Fewer pairs but same accuracy → stricter model, stylistic choice only

Legal-BERT out of the box tends to produce a degenerate embedding space
(most tags look similar to most others). This shows up as very high pair
counts with no downstream improvement.


## 8. Atom classifier

Given an atom-shaped input, rank all corpus atoms by similarity and predict
an article. Score composition:

- **exact_score** : sum of IDF weights of tags shared with the query
- **semantic_score** : sum of best semantic-similarity matches for tags not exactly matched
- **combined_score** = `alpha * exact_score + (1 - alpha) * semantic_score`

`alpha=0.5` weights both channels equally.


### 8.1 Classify an existing atom

Outputs Justification for classification.


In [31]:
query = df[df['atom_id'] == '7:454(2).a'].iloc[0].to_dict()
result = classify_atom(query, df, weights=weights_idf, sim_pairs=sim_pairs, alpha=0.5, top_k=5)
print(f"Predicted article: {result['predicted_article']}")
result['top_matches']


Predicted article: 7:454


,atom_id,article_id,combined_score,exact_score,semantic_score
0,7:455(1).a,7:455,5.8358,10.9474,0.7242
1,7:454(1).a,7:454,4.1033,8.2065,0.0000
2,7:454(1).b,7:454,2.7328,5.4657,0.0000
3,7:455(2).a,7:455,2.2882,3.9776,0.5987
4,7:454(1).d,7:454,2.1145,4.2289,0.0000


In [18]:
# Why did the top match win? Break down its contributing exact and semantic matches.
top_row = result['scores_by_atom'].iloc[0]
print(f"Top match: {top_row['atom_id']}")
print(f"  combined_score = {top_row['combined_score']}")
print(f"  exact_score    = {top_row['exact_score']}")
print(f"  semantic_score = {top_row['semantic_score']}")
print()
print('Exact matches (field, tag, weight):')
for m in top_row['exact_matches']:
    print(f'  {m}')
print()
print('Semantic matches (field, query_tag, candidate_tag, similarity):')
for m in top_row['semantic_matches']:
    print(f'  {m}')


Top match: 7:455(1).a
  combined_score = 5.8358
  exact_score    = 10.9474
  semantic_score = 0.7242

Exact matches (field, tag, weight):
  ('actors', 'care provider', 2.0477)
  ('actors', 'patient', 1.2368)
  ('legal_relations', 'treatment agreement', 2.7408)
  ('residual', 'file', 2.1812)
  ('residual', 'request', 2.7408)

Semantic matches (field, query_tag, candidate_tag, similarity):
  ('temporal', 'upon request', 'after request', 0.7242)


### 8.2 Classify a hand-written query

A hypothetical atom phrased as a paraphrase of 7:454(1).a. If the classifier
works, that atom should rank near the top.


In [32]:
hand_query = {
    'atom_id': 'HAND_QUERY',
    'article_id': 'UNKNOWN',
    'actors': ['care provider', 'patient'],
    'legal_relations': ['treatment agreement'],
    'acts': ['sets up'],
    'residual': ['file', 'treatment'],
    'temporal': [], 'explicit_references': [], 'hierarchies': [], 'geographical_domain': [],
}
result = classify_atom(hand_query, df, weights=weights_idf, sim_pairs=sim_pairs, alpha=0.5, top_k=5)
print(f"Predicted article: {result['predicted_article']}")
result['top_matches']


Predicted article: 7:454


,atom_id,article_id,combined_score,exact_score,semantic_score
0,7:454(1).a,7:454,7.5372,15.0745,0.0
1,7:454(2).a,7:454,4.1033,8.2065,0.0
2,7:455(1).a,7:455,4.1033,8.2065,0.0
3,7:454(1).b,7:454,2.7328,5.4657,0.0
4,7:454(1).d,7:454,2.1145,4.2289,0.0


### 8.3 Leave-one-out evaluation

For each atom: remove it, classify against the rest, record whether the
correct article appears at rank 1 and in top-k, plus reciprocal rank.


In [33]:
eval_df, summary = evaluate_classifier_loo(df, weights=weights_idf, sim_pairs=sim_pairs, alpha=0.5, top_k=3)
print('Overall:')
for k, v in summary.items():
    print(f'  {k:25s} {v}')
print()
eval_df[['atom_id', 'true_article', 'top1_match', 'top1_hit', 'rank']]


Overall:
  n_atoms                   61
  top1_accuracy             0.2787
  top3_accuracy             0.4262
  mean_reciprocal_rank      0.3497



,atom_id,true_article,top1_match,top1_hit,rank
0,6:162(1).a,6:162,6:162,True,1.0
1,6:162(2).a,6:162,6:162,True,1.0
2,6:162(2).b,6:162,6:162,True,1.0
3,6:162(3).a,6:162,6:75,False,2.0
4,6:170(1).a,6:170,6:170,True,1.0
...,...,...,...,...,...
56,synth:14(1).a,synth:14,synth:2,False,NaN
57,synth:14(1).b,synth:14,synth:13,False,NaN
58,synth:15(1).a,synth:15,synth:3,False,NaN
59,synth:15(1).b,synth:15,synth:15,True,1.0


### 8.4 Per-article breakdown


In [21]:
per_article = eval_df.groupby('true_article').agg(
    n_atoms=('atom_id', 'size'),
    top1_accuracy=('top1_hit', 'mean'),
    topk_accuracy=('topk_hit', 'mean'),
    mrr=('reciprocal_rank', 'mean'),
).round(3)
per_article


,n_atoms,top1_accuracy,topk_accuracy,mrr
true_article,,,,
6:162,4,0.750,1.000,0.875
6:170,3,1.000,1.000,1.000
6:74,3,0.000,1.000,0.444
6:75,1,0.000,0.000,0.000
6:76,1,0.000,0.000,0.000
7:454,8,0.625,1.000,0.812
7:455,3,0.333,0.333,0.333
synth:1,2,0.000,0.000,0.000
synth:10,3,0.000,0.000,0.000


### 8.5 Alpha and top-k sweep

Grid over `alpha` (mixing exact vs semantic) and `top_k` (retrieval cutoff).
`alpha=0` uses semantic only; `alpha=1` uses exact only.


In [22]:
sweep_rows = []
for a in [0.0, 0.25, 0.5, 0.75, 1.0]:
    for k in [3, 5]:
        _, s = evaluate_classifier_loo(df, weights=weights_idf, sim_pairs=sim_pairs, alpha=a, top_k=k)
        sweep_rows.append({'alpha': a, 'top_k': k, **s})
pd.DataFrame(sweep_rows)


,alpha,top_k,n_atoms,top1_accuracy,top3_accuracy,mean_reciprocal_rank,top5_accuracy
0,0.00,3,61,0.1311,0.2131,0.1667,NaN
1,0.00,5,61,0.1311,NaN,0.1839,0.2951
2,0.25,3,61,0.2623,0.4426,0.3470,NaN
3,0.25,5,61,0.2623,NaN,0.3536,0.4754
4,0.50,3,61,0.2787,0.4262,0.3497,NaN
5,0.50,5,61,0.2787,NaN,0.3604,0.4754
6,0.75,3,61,0.2787,0.4262,0.3497,NaN
7,0.75,5,61,0.2787,NaN,0.3596,0.4754
8,1.00,3,61,0.2623,0.4098,0.3333,NaN
9,1.00,5,61,0.2623,NaN,0.3399,0.4426


### 8.6 Misclassifications

Atoms where the correct article didn't reach rank 1. Errors that look like
sibling articles are usually intuitive; errors that jump across topics are
worth digging into.


In [23]:
misses = eval_df[~eval_df['top1_hit']]
misses[['atom_id', 'true_article', 'top1_match', 'rank']]


,atom_id,true_article,top1_match,rank
3,6:162(3).a,6:162,6:75,2.0
7,6:74(1).a,6:74,6:76,3.0
8,6:74(1).b,6:74,6:75,2.0
9,6:74(2).a,6:74,6:76,2.0
10,6:75(1).a,6:75,6:162,NaN
11,6:76(1).a,6:76,6:74,NaN
13,7:454(1).b,7:454,7:455,2.0
15,7:454(1).d,7:454,7:455,2.0
17,7:454(2).a,7:454,7:455,2.0
20,7:455(1).a,7:455,7:454,NaN


## 9. Legal hierarchy

Every article belongs somewhere in the Dutch civil code tree:

    Rechtsgebied  (private_law / public_law)
        Wetboek       (BW, Sr, Sv, ...)
            Boek          (1-10 for BW)
                Titel         (e.g. 7.7 - Opdracht)
                    Afdeling      (e.g. 7.7.5 - WGBO)
                        Artikel       (e.g. 7:454)

`src/data/hierarchy.py` encodes this tree and parses article IDs into their
taxonomy path. Book 7 is fully populated because that is where the WGBO
corpus lives. Other books have names but no title-level breakdown yet.


### 9.1 Parse taxonomy for the corpus

Overview of every atom's taxonomy (Which article, book, titel and afdeling it belongs to).


In [24]:
from src.data.hierarchy import (
    parse_article_id, add_taxonomy_columns, tree_summary,
    build_hierarchy_edges, evaluate_at_levels,
)

df_tax = add_taxonomy_columns(df)
df_tax[['atom_id', 'article_id', 'boek', 'titel', 'afdeling', 'taxonomy_path']].head(12)


,atom_id,article_id,boek,titel,afdeling,taxonomy_path
0,6:162(1).a,6:162,Boek 6,6.3,6.3.1,private_law > BW > Boek 6 > Titel 6.3 > Afd 6.3.1 > Art 6:162
1,6:162(2).a,6:162,Boek 6,6.3,6.3.1,private_law > BW > Boek 6 > Titel 6.3 > Afd 6.3.1 > Art 6:162
2,6:162(2).b,6:162,Boek 6,6.3,6.3.1,private_law > BW > Boek 6 > Titel 6.3 > Afd 6.3.1 > Art 6:162
3,6:162(3).a,6:162,Boek 6,6.3,6.3.1,private_law > BW > Boek 6 > Titel 6.3 > Afd 6.3.1 > Art 6:162
4,6:170(1).a,6:170,Boek 6,6.3,6.3.2,private_law > BW > Boek 6 > Titel 6.3 > Afd 6.3.2 > Art 6:170
5,6:170(1).b,6:170,Boek 6,6.3,6.3.2,private_law > BW > Boek 6 > Titel 6.3 > Afd 6.3.2 > Art 6:170
6,6:170(1).c,6:170,Boek 6,6.3,6.3.2,private_law > BW > Boek 6 > Titel 6.3 > Afd 6.3.2 > Art 6:170
7,6:74(1).a,6:74,Boek 6,6.1,6.1.9,private_law > BW > Boek 6 > Titel 6.1 > Afd 6.1.9 > Art 6:74
8,6:74(1).b,6:74,Boek 6,6.1,6.1.9,private_law > BW > Boek 6 > Titel 6.1 > Afd 6.1.9 > Art 6:74
9,6:74(2).a,6:74,Boek 6,6.1,6.1.9,private_law > BW > Boek 6 > Titel 6.1 > Afd 6.1.9 > Art 6:74


### 9.2 Tree summary
How many atoms formed at every level of the structure


In [25]:
tree_summary(df_tax)


,level,value,n_atoms
0,rechtsgebied,synthetic,38
1,rechtsgebied,private_law,23
2,wetboek,LMC,38
3,wetboek,BW,23
4,boek,(none),38
5,boek,Boek 6,12
6,boek,Boek 7,11
7,titel,(none),38
8,titel,7.7,11
9,titel,6.3,7


### 9.3 Containment edges

The hierarchy adds a new edge type  directed parent-child containment
edges from atoms up through the tree. These sit alongside the tag-overlap
and semantic edges from Sections 3 and 6.

So far all the edges are due to some similairty between atoms these new edges are different they mimic the structure of the Dutch Civil Code. They say "this thing is inside that thing." They're the edges of the tree structure itself.


In [26]:
hier_edges = build_hierarchy_edges(df_tax)
print(f'{len(hier_edges)} containment edges')
print()
print('Edges by level:')
print(hier_edges['level'].value_counts().to_string())


94 containment edges

Edges by level:
level
atom_to_article            61
article_to_wetboek         15
article_to_afdeling         7
afdeling_to_titel           4
titel_to_boek               3
boek_to_wetboek             2
wetboek_to_rechtsgebied     2


In [27]:
# Sample the edges — one row per unique containment relation
hier_edges.drop_duplicates(subset=['source', 'target'])


,source,target,edge_kind,level
0,6:162(1).a,6:162,contains,atom_to_article
1,6:162,6.3.1,contains,article_to_afdeling
2,6.3.1,6.3,contains,afdeling_to_titel
3,6.3,Boek 6,contains,titel_to_boek
4,Boek 6,BW,contains,boek_to_wetboek
...,...,...,...,...
89,synth:14(1).b,synth:14,contains,atom_to_article
90,synth:15(1).a,synth:15,contains,atom_to_article
91,synth:15,LMC,contains,article_to_wetboek
92,synth:15(1).b,synth:15,contains,atom_to_article


### 9.4 Hierarchical accuracy

Leave-one-out with accuracy reported at each level of the tree, not just at
the article level. For a corpus concentrated in one afdeling (WGBO), the
book/titel/afdeling accuracies are trivially high because there is little
variation to distinguish. This becomes meaningful once atoms from other
sections (e.g. Book 6 obligations) are added.


In [28]:
eval_hier, summary_hier = evaluate_at_levels(
    df_tax, weights=weights_idf, sim_pairs=sim_pairs, alpha=0.5, top_k=3
)
print('Accuracy by level (higher = coarser prediction):')
for k, v in summary_hier.items():
    print(f'  {k:30s} {v}')


Accuracy by level (higher = coarser prediction):
  rechtsgebied_accuracy          0.836
  wetboek_accuracy               0.836
  boek_accuracy                  0.361
  titel_accuracy                 0.328
  afdeling_accuracy              0.328
  article_id_accuracy            0.279


In [29]:
# Per-atom breakdown: where did the prediction land in the tree?
eval_hier[['atom_id',
           'rechtsgebied_true', 'rechtsgebied_pred', 'rechtsgebied_hit',
           'boek_true',         'boek_pred',         'boek_hit',
           'article_id_true',   'article_id_pred',   'article_id_hit']]


,atom_id,rechtsgebied_true,rechtsgebied_pred,rechtsgebied_hit,boek_true,boek_pred,boek_hit,article_id_true,article_id_pred,article_id_hit
0,6:162(1).a,private_law,private_law,True,Boek 6,Boek 6,True,6:162,6:162,True
1,6:162(2).a,private_law,private_law,True,Boek 6,Boek 6,True,6:162,6:162,True
2,6:162(2).b,private_law,private_law,True,Boek 6,Boek 6,True,6:162,6:162,True
3,6:162(3).a,private_law,private_law,True,Boek 6,Boek 6,True,6:162,6:75,False
4,6:170(1).a,private_law,private_law,True,Boek 6,Boek 6,True,6:170,6:170,True
...,...,...,...,...,...,...,...,...,...,...
56,synth:14(1).a,synthetic,synthetic,True,None,None,False,synth:14,synth:2,False
57,synth:14(1).b,synthetic,synthetic,True,None,None,False,synth:14,synth:13,False
58,synth:15(1).a,synthetic,synthetic,True,None,None,False,synth:15,synth:3,False
59,synth:15(1).b,synthetic,synthetic,True,None,None,False,synth:15,synth:15,True
